# Travel Planner (Multi-Agent System)

**Scenario:** This project uses a LangGraph multi-agent workflow to research and plan travel itineraries. It features a **Researcher agent** that gathers weather and pricing data via custom tools, and a **Planner agent** that uses that data to draft a structured, day-by-day itinerary.

In [1]:
!pip install -q langchain langchain-openai langgraph

import os
import uuid
from pathlib import Path
from typing import Annotated, Dict, List, Literal, Optional, TypedDict

from IPython.display import Image, display
from google.colab import userdata
from pydantic import BaseModel, Field, SecretStr

from langchain_core.messages import AIMessage, BaseMessage, HumanMessage, SystemMessage
from langchain_core.prompts import PromptTemplate
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI

from langgraph.checkpoint.memory import InMemorySaver
from langgraph.graph import END, START, StateGraph
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode
from langgraph.types import Command, interrupt

try:
    openai_api_key = SecretStr(userdata.get("OPENAI_API_KEY"))
except userdata.SecretNotFoundError:
    print("Error: 'OPENAI_API_KEY' is missing in Google Colab Secrets.")
    raise

try:
    langsmith_key = userdata.get("LANGSMITH_API_KEY")
    if langsmith_key:
        os.environ["LANGSMITH_TRACING"] = "true"
        os.environ["LANGSMITH_API_KEY"] = langsmith_key
        os.environ["LANGSMITH_PROJECT"] = "Travel_Planner"
except userdata.SecretNotFoundError:
    pass

def display_graph(runnable, output_png: Path = Path("workflow_graph.png")) -> None:
    """Generates and displays a Mermaid diagram of the compiled graph."""

    try:
        with output_png.open(mode="wb") as file:
            file.write(runnable.get_graph(xray=True).draw_mermaid_png())
        display(Image(output_png, format="png"))
    except Exception as e:
        print(f"Failed to generate graph visualization: {e}")

In [2]:
class HumanFeedback(TypedDict):
    """Stores the approval status and comments from the human reviewer."""

    approved: bool
    comment: Optional[str]

class DailyPlan(BaseModel):
    """Structured representation of a single day in the itinerary."""

    day_number: int = Field(description="The day number of the trip (e.g., 1, 2).")
    activities: str = Field(description="Detailed plan of activities for this day.")
    accommodation: str = Field(description="Hotel or accommodation details for the night.")

class TravelState(TypedDict):
    """Maintains the state of the travel planning workflow."""

    user_request: str
    target_destination: str
    research_data: str
    itinerary: List[dict]
    messages: Annotated[List[BaseMessage], add_messages]
    human_feedback: HumanFeedback
    revision_cycles: int

In [3]:
@tool
def get_weather_info(destination: str, month: str) -> str:
    """Returns the average temperatures and weather conditions for a given destination and month."""

    if not destination or not month:
        return "Error: Destination and month must be provided."

    weather_db = {
        "Paris": "Average 15°C, expect light rain and mild breezes.",
        "Tokyo": "Average 22°C, sunny and warm, ideal for sightseeing.",
        "Rome": "Average 26°C, very sunny and hot.",
        "London": "Average 13°C, overcast with frequent showers."
    }

    dest_lower = str(destination).lower()
    for key, forecast in weather_db.items():
        if key in dest_lower:
            return f"Weather in {destination} during {month}: {forecast}"

    return f"Weather in {destination} during {month}: Average 20°C, mostly clear."


@tool
def search_travel_options(destination: str, budget_level: str) -> str:
    """Searches for flight and hotel options based on destination and budget (low, medium, high)."""

    if not destination or not budget_level:
        return "Error: Destination and budget level must be provided."

    budget = str(budget_level).lower()

    if "low" in budget:
        return f"Low-budget options for {destination}: Low-cost flight (approx. $150). Accommodation: Hostel or 2-star hotel ($40/night)."
    elif "high" in budget or "luxury" in budget:
        return f"Luxury options for {destination}: Direct premium flight (approx. $800+). Accommodation: 4/5-star hotel ($250+/night)."
    else:
        return f"Standard options for {destination}: Economy flight (approx. $300). Accommodation: 3-star hotel ($120/night)."


TOOLS = [get_weather_info, search_travel_options]

In [4]:
class ItineraryResponse(BaseModel):
    """Wrapper model to ensure valid structured output from LangChain."""

    days: List[DailyPlan] = Field(description="A list containing the daily itinerary plans.")

researcher_model = ChatOpenAI(model="gpt-5-nano", api_key=openai_api_key, reasoning_effort="low").bind_tools(TOOLS)
planner_model = ChatOpenAI(model="gpt-5.4-mini", api_key=openai_api_key, reasoning_effort="low").with_structured_output(ItineraryResponse)

RESEARCHER_PROMPT = """You are an expert Travel Researcher.
Your objective is to gather accurate weather data, flight options, and accommodation details for the requested destination.
Always use your provided tools to fetch the data. Do not guess or make up prices.

User Request: {request}"""

PLANNER_PROMPT = """You are a Senior Itinerary Planner.
Based on the raw research data (provided in the conversation history) and the user's initial request, draft a structured, day-by-day travel itinerary.

Original Request: {request}

Human Feedback for Revision (if any):
{feedback}

Ensure your output strictly follows the required format. Address any human feedback directly in your new draft."""

def researcher_node(state: TravelState) -> dict:
    request = state.get("user_request", "")
    sys_msg = SystemMessage(content=RESEARCHER_PROMPT.format(request=request))
    messages = [sys_msg] + state.get("messages", [])

    response = researcher_model.invoke(messages)
    return {"messages": [response]}

def planner_node(state: TravelState) -> dict:
    request = state.get("user_request", "")
    feedback_dict = state.get("human_feedback", {})
    feedback_text = feedback_dict.get("comment", "None") if feedback_dict else "None"

    sys_msg = SystemMessage(content=PLANNER_PROMPT.format(request=request, feedback=feedback_text))
    messages = [sys_msg] + state.get("messages", [])

    structured_plan = planner_model.invoke(messages)

    draft_summary = "\n".join([f"Day {p.day_number}: {p.activities}" for p in structured_plan.days])
    ai_memory_message = AIMessage(content=f"Here is my drafted itinerary:\n{draft_summary}")

    itinerary_dicts = [p.model_dump() for p in structured_plan.days]

    return {"itinerary": itinerary_dicts, "messages": [ai_memory_message]}

In [5]:
def route_researcher(state: TravelState) -> str:
    """
    Routes the graph based on the researcher's output.
    If a tool was called, route to 'tools'. Otherwise, route to 'planner'.
    """

    messages = state.get("messages", [])
    if not messages:
        return "planner"

    last_message = messages[-1]

    if hasattr(last_message, 'tool_calls') and last_message.tool_calls:
        return "tools"

    return "planner"

def human_review_node(state: TravelState) -> dict:
    """
    Interrupts the workflow to request human feedback on the drafted itinerary.
    """

    cycles = state.get("revision_cycles", 0)

    human_input = interrupt(f"Please review the drafted itinerary. (Revision cycle: {cycles})")

    feedback = {
        "approved": human_input.get("approved", True),
        "comment": human_input.get("comment", None)
    }

    return {"human_feedback": feedback, "revision_cycles": cycles + 1}

def route_review(state: TravelState) -> str:
    """
    Determines whether the workflow should end or loop back to the planner.
    """

    feedback = state.get("human_feedback", {})

    if feedback.get("approved", True):
        return END

    return "planner"


graph_builder = StateGraph(TravelState)

graph_builder.add_node("researcher", researcher_node)
graph_builder.add_node("tools", ToolNode(TOOLS))
graph_builder.add_node("planner", planner_node)
graph_builder.add_node("human_review", human_review_node)

graph_builder.add_edge(START, "researcher")
graph_builder.add_conditional_edges("researcher", route_researcher, ["tools", "planner"])
graph_builder.add_edge("tools", "researcher")
graph_builder.add_edge("planner", "human_review")
graph_builder.add_conditional_edges("human_review", route_review, ["planner", END])

memory_saver = InMemorySaver()
travel_graph = graph_builder.compile(checkpointer=memory_saver)

 # display_graph(travel_graph)

In [6]:
def execute_workflow(user_request: str, simulate_rejection: bool = False):
    """
    Initializes the graph, handles the interrupt, and resumes execution.
    Creates a unique thread_id for each execution to ensure isolated memory states.
    Includes error handling and optimized DRY logic for human review.
    """

    thread_id = uuid.uuid4().hex
    config = {"configurable": {"thread_id": thread_id}}

    print(f"\n{'-' * 50}")
    print(f"[SYSTEM] Initializing request: {user_request}")
    print(f"{'-' * 50}")

    initial_state = {"user_request": user_request, "revision_cycles": 0}

    try:
        state = travel_graph.invoke(initial_state, config=config)

        if state and "__interrupt__" in state:
            print("[HITL] Execution paused. Awaiting human validation...")

            if simulate_rejection:
                print("[HITL] Input received: REJECTED. Feedback provided.")
                human_response = {"approved": False, "comment": "Please add more specific local food recommendations."}
                state = travel_graph.invoke(Command(resume=human_response), config=config)
                print("[HITL] Rework cycle completed. Awaiting secondary validation...")

            print("[HITL] Input received: APPROVED.")
            final_response = {"approved": True, "comment": ""}
            state = travel_graph.invoke(Command(resume=final_response), config=config)

        final_itinerary = state.get("itinerary", [])
        print("\n[RESULT] Final Approved Itinerary:")

        if not final_itinerary:
            print("  Warning: No itinerary generated.")
        else:
            for plan in final_itinerary:
                print(f"  Day {plan['day_number']}:")
                print(f"    Accommodation: {plan['accommodation']}")
                print(f"    Activities:    {plan['activities']}")
                print("    ...")

    except Exception as e:
        print(f"\n[ERROR] The workflow encountered a critical issue: {str(e)}")
        print("[SYSTEM] Please check your API limits or network connection.")

In [7]:
print("Executing Test Case 1: Standard Approval")
execute_workflow("I want a low budget 2-day trip to Paris.")

Executing Test Case 1: Standard Approval

--------------------------------------------------
[SYSTEM] Initializing request: I want a low budget 2-day trip to Paris.
--------------------------------------------------
[HITL] Execution paused. Awaiting human validation...
[HITL] Input received: APPROVED.

[RESULT] Final Approved Itinerary:
  Day 1:
    Accommodation: Low-budget hostel or 2-star hotel in a central, well-connected area such as Montmartre, Bastille, Republique, or near Gare du Nord.
    Activities:    Morning: Arrive in Paris and check into a low-cost hostel or 2-star hotel. Late morning: Take a free walking route along the Seine, including Île de la Cité and the outside of the Louvre courtyard. Lunch: Grab an inexpensive sandwich or bakery meal. Afternoon: Explore Montmartre on foot, including the Sacré-Cœur area and Place du Tertre. Evening: Head to Trocadéro or a Seine riverside spot for budget-friendly sunset views of the Eiffel Tower, then have a low-cost dinner at a ca

In [8]:
print("Executing Test Case 2: Human-in-the-Loop Rework Cycle")
execute_workflow("Plan a standard 3-day trip to Tokyo.", simulate_rejection=True)

Executing Test Case 2: Human-in-the-Loop Rework Cycle

--------------------------------------------------
[SYSTEM] Initializing request: Plan a standard 3-day trip to Tokyo.
--------------------------------------------------
[HITL] Execution paused. Awaiting human validation...
[HITL] Input received: REJECTED. Feedback provided.
[HITL] Rework cycle completed. Awaiting secondary validation...
[HITL] Input received: APPROVED.

[RESULT] Final Approved Itinerary:
  Day 1:
    Accommodation: 3-star hotel in the Shibuya, Shinjuku, or Tokyo Station area
    Activities:    Morning: Arrive in Tokyo and check in, then start in Shibuya with Shibuya Crossing and the Hachiko Statue. Browse Shibuya Center Street for shopping and easy lunch options. For a local food recommendation, try a bowl of tonkotsu ramen, a beef gyudon set, or a crispy pork katsu sandwich at a casual Shibuya spot. Afternoon: Head to Harajuku to walk Takeshita Street, then visit Meiji Shrine for a peaceful cultural break. Contin

In [9]:
print("Executing Test Case 3: Luxury Segment")
execute_workflow("Luxury 2-day weekend in Rome.")

Executing Test Case 3: Luxury Segment

--------------------------------------------------
[SYSTEM] Initializing request: Luxury 2-day weekend in Rome.
--------------------------------------------------
[HITL] Execution paused. Awaiting human validation...
[HITL] Input received: APPROVED.

[RESULT] Final Approved Itinerary:
  Day 1:
    Accommodation: Stay at a central 4- or 5-star luxury hotel in Rome, ideally near the Vatican, Piazza di Spagna, or the Centro Storico, with spa or wellness facilities and high-end service.
    Activities:    Arrive in Rome and transfer privately to a central 5-star hotel for check-in and refreshment. Begin with a skip-the-line private guided visit to the Vatican Museums and St. Peter’s Basilica, with the option to add the Vatican Gardens for a more exclusive experience. Enjoy lunch at a Michelin-starred restaurant or an upscale Roman trattoria in the Prati or Centro Storico area. In the afternoon, take a private transfer for a VIP tour of the Colosseum, 

In [10]:
print("Executing Test Case 4: Weather Context Adaptation")
execute_workflow("I need a low budget 1-day trip to London in November.")

Executing Test Case 4: Weather Context Adaptation

--------------------------------------------------
[SYSTEM] Initializing request: I need a low budget 1-day trip to London in November.
--------------------------------------------------
[HITL] Execution paused. Awaiting human validation...
[HITL] Input received: APPROVED.

[RESULT] Final Approved Itinerary:
  Day 1:
    Accommodation: Not required for a 1-day trip. If staying overnight, choose a hostel or 2-star hotel in central London for about $40 per night.
    Activities:    Morning: Begin in central London with free walking sights around Westminster, including exterior views of Big Ben, Parliament, and Westminster Abbey. Continue a scenic walk along the South Bank for river views and a budget-friendly start to the day. Midday: Keep lunch low-cost with street food or a supermarket meal near Borough Market or the South Bank. If you want a free indoor option, visit the British Museum or Tate Modern permanent collections. Afternoon: 

In [11]:
print("Executing Test Case 5: Budget Constraint Rework")
execute_workflow("High budget 2-day vacation in Paris.", simulate_rejection=True)

Executing Test Case 5: Budget Constraint Rework

--------------------------------------------------
[SYSTEM] Initializing request: High budget 2-day vacation in Paris.
--------------------------------------------------
[HITL] Execution paused. Awaiting human validation...
[HITL] Input received: REJECTED. Feedback provided.
[HITL] Rework cycle completed. Awaiting secondary validation...
[HITL] Input received: APPROVED.

[RESULT] Final Approved Itinerary:
  Day 1:
    Accommodation: Le Bristol Paris, an ultra-luxury 5-star stay in the 8th arrondissement, ideal for a high-budget Paris escape.
    Activities:    Arrive in Paris and enjoy a private airport transfer to your luxury hotel. Start with a late-morning brunch in the 8th arrondissement at Le Café Marly for a classic Parisian setting, or L’Avenue for a chic local favorite. Then take a private guided visit to the Eiffel Tower with priority access for panoramic city views. Continue with an elegant Seine river cruise on a private or pr